In [13]:
import sqlite3
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    accuracy_score
)

from xgboost import XGBClassifier

DB_PATH = "../data/soccer.db"

conn = sqlite3.connect(DB_PATH)

In [14]:
TRAIN_MAX_SEASON = 2023
TEST_MIN_SEASON = 2024

EUROPEAN_LEAGUES = [
    1,   # Premier League
    2,  # La Liga
    3,  # Serie A
    4,   # Bundesliga
    5    # Ligue 1
]

LEAGUE_MAP = {
    39: "Premier League",
    140: "La Liga",
    135: "Serie A",
    78: "Bundesliga",
    61: "Ligue 1"
}

In [15]:
matches_query = """
SELECT
    *
FROM matches
"""

matches_df = pd.read_sql(
    matches_query,
    conn
)

matches_df = matches_df[
    matches_df["competition_id"].isin(EUROPEAN_LEAGUES)
].copy()

context_query = """
SELECT
    *
FROM match_context
"""

context_df = pd.read_sql(
    context_query,
    conn
)

context_df = context_df.drop(columns=["id"])

matches_features = matches_df.merge(
    context_df,
    left_on="id",
    right_on="match_id",
    how="left"
)

matches_features = matches_features.sort_values(
    by="date"
).reset_index(drop=True)

In [16]:
matches_features["home_elo"] = 1500.0
matches_features["away_elo"] = 1500.0

elo_ratings = {}

K_FACTOR = 20

for idx, row in matches_features.iterrows():

    home_team = row["home_team_id"]
    away_team = row["away_team_id"]

    home_elo = elo_ratings.get(home_team, 1500)
    away_elo = elo_ratings.get(away_team, 1500)

    matches_features.at[idx, "home_elo"] = home_elo
    matches_features.at[idx, "away_elo"] = away_elo

    expected_home = (
        1 /
        (
            1 +
            10 ** (
                (away_elo - home_elo) / 400
            )
        )
    )

    expected_away = 1 - expected_home

    if row["home_goals"] > row["away_goals"]:

        actual_home = 1
        actual_away = 0

    elif row["home_goals"] < row["away_goals"]:

        actual_home = 0
        actual_away = 1

    else:

        actual_home = 0.5
        actual_away = 0.5

    new_home_elo = (
        home_elo +
        K_FACTOR *
        (
            actual_home -
            expected_home
        )
    )

    new_away_elo = (
        away_elo +
        K_FACTOR *
        (
            actual_away -
            expected_away
        )
    )

    elo_ratings[home_team] = new_home_elo
    elo_ratings[away_team] = new_away_elo

In [17]:
matches_features["home_win"] = (
    matches_features["home_goals"] >
    matches_features["away_goals"]
)

matches_features["away_win"] = (
    matches_features["away_goals"] >
    matches_features["home_goals"]
)

matches_features["is_draw"] = (
    matches_features["home_goals"] ==
    matches_features["away_goals"]
)

matches_features["home_dnb"] = (
    matches_features["home_goals"] >
    matches_features["away_goals"]
)

matches_features["away_dnb"] = (
    matches_features["away_goals"] >
    matches_features["home_goals"]
)

matches_features["match_result_3class"] = np.select(
    [
        matches_features["home_win"],
        matches_features["is_draw"],
        matches_features["away_win"]
    ],
    [
        0,
        1,
        2
    ]
)

matches_features["round_number"] = (
    matches_features["round"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

In [18]:
matches_features["home_draw_rate_last_10"] = (
    matches_features
    .groupby("home_team_id")["is_draw"]
    .transform(
        lambda x:
        x.shift(1).rolling(10, min_periods=1).mean()
    )
)

matches_features["away_draw_rate_last_10"] = (
    matches_features
    .groupby("away_team_id")["is_draw"]
    .transform(
        lambda x:
        x.shift(1).rolling(10, min_periods=1).mean()
    )
)

matches_features["home_rolling_scored"] = (
    matches_features
    .groupby("home_team_id")["home_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

matches_features["home_rolling_conceded"] = (
    matches_features
    .groupby("home_team_id")["away_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

matches_features["away_rolling_scored"] = (
    matches_features
    .groupby("away_team_id")["away_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

matches_features["away_rolling_conceded"] = (
    matches_features
    .groupby("away_team_id")["home_goals"]
    .transform(
        lambda x:
        x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [19]:
matches_features["elo_diff"] = (
    matches_features["home_elo"] -
    matches_features["away_elo"]
).abs()

matches_features["attack_diff"] = (
    matches_features["home_rolling_scored"] -
    matches_features["away_rolling_scored"]
).abs()

matches_features["defense_diff"] = (
    matches_features["home_rolling_conceded"] -
    matches_features["away_rolling_conceded"]
).abs()

matches_features["combined_scoring"] = (
    matches_features["home_rolling_scored"] +
    matches_features["away_rolling_scored"]
)

matches_features["combined_conceded"] = (
    matches_features["home_rolling_conceded"] +
    matches_features["away_rolling_conceded"]
)

matches_features["total_balance"] = (
    matches_features["attack_diff"] +
    matches_features["defense_diff"]
)

matches_features["points_diff_abs"] = (
    matches_features["points_diff"]
).abs()

matches_features["position_diff_abs"] = (
    matches_features["position_diff"]
).abs()

matches_features["draw_rate_diff"] = (
    matches_features["home_draw_rate_last_10"] -
    matches_features["away_draw_rate_last_10"]
).abs()

matches_features["draw_rate_sum"] = (
    matches_features["home_draw_rate_last_10"] +
    matches_features["away_draw_rate_last_10"]
)

matches_features["rest_days_diff"] = (
    matches_features["home_rest_days"] -
    matches_features["away_rest_days"]
).abs()

matches_features["coach_tenure_diff"] = (
    matches_features["home_coach_tenure_days"] -
    matches_features["away_coach_tenure_days"]
).abs()

matches_features["home_advantage_strength"] = (
    matches_features["home_rolling_scored"] -
    matches_features["away_rolling_conceded"]
)

matches_features["away_advantage_strength"] = (
    matches_features["away_rolling_scored"] -
    matches_features["home_rolling_conceded"]
)

matches_features["strength_gap"] = (
    matches_features["home_advantage_strength"] -
    matches_features["away_advantage_strength"]
).abs()

matches_features["table_pressure_match"] = (
    matches_features["home_title_race"] +
    matches_features["away_title_race"] +
    matches_features["home_europe_race"] +
    matches_features["away_europe_race"] +
    matches_features["home_relegation_risk"] +
    matches_features["away_relegation_risk"]
)

matches_features["low_scoring_profile"] = (
    matches_features["combined_scoring"] <= 2.2
).astype(int)

matches_features["balanced_match_flag"] = (
    (
        matches_features["elo_diff"] <= 50
    ) &
    (
        matches_features["position_diff_abs"] <= 3
    ) &
    (
        matches_features["points_diff_abs"] <= 6
    )
).astype(int)

In [20]:
for col in multiclass_features:

    null_count = matches_features[col].isna().sum()

    print(col, "->", null_count)

home_elo -> 0
away_elo -> 0
home_rolling_scored -> 183
away_rolling_scored -> 183
home_rolling_conceded -> 183
away_rolling_conceded -> 183
home_draw_rate_last_10 -> 163
away_draw_rate_last_10 -> 163
home_position -> 136
away_position -> 136
points_diff -> 136
position_diff -> 136
home_rest_days -> 18067
away_rest_days -> 18067
home_coach_tenure_days -> 5167
away_coach_tenure_days -> 5157
home_title_race -> 136
away_title_race -> 136
home_europe_race -> 136
away_europe_race -> 136
home_relegation_risk -> 136
away_relegation_risk -> 136
elo_diff -> 0
attack_diff -> 253
defense_diff -> 253
combined_scoring -> 253
combined_conceded -> 253
total_balance -> 253
points_diff_abs -> 136
position_diff_abs -> 136
draw_rate_diff -> 227
draw_rate_sum -> 227
rest_days_diff -> 18067
coach_tenure_diff -> 8518
home_advantage_strength -> 253
away_advantage_strength -> 253
strength_gap -> 253
table_pressure_match -> 136
low_scoring_profile -> 0
balanced_match_flag -> 0
round_number -> 29


In [24]:
print(train_df.shape)

print(test_df.shape)

print(y_train.value_counts())

print(y_test.value_counts())

(14211, 64)
(3461, 64)
match_result_3class
0    6291
2    4363
1    3557
Name: count, dtype: int64
match_result_3class
0    1486
2    1102
1     873
Name: count, dtype: int64


In [23]:
# ===========================================
# MULTICLASS FEATURES
# ===========================================

multiclass_features = [

    "home_elo",
    "away_elo",

    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_draw_rate_last_10",
    "away_draw_rate_last_10",

    "home_position",
    "away_position",

    "points_diff",
    "position_diff",

    "home_title_race",
    "away_title_race",

    "home_europe_race",
    "away_europe_race",

    "home_relegation_risk",
    "away_relegation_risk",

    "elo_diff",
    "attack_diff",
    "defense_diff",

    "combined_scoring",
    "combined_conceded",

    "total_balance",

    "points_diff_abs",
    "position_diff_abs",

    "draw_rate_diff",
    "draw_rate_sum",

    "home_advantage_strength",
    "away_advantage_strength",

    "strength_gap",

    "table_pressure_match",

    "low_scoring_profile",

    "balanced_match_flag",

    "round_number"
]

# ===========================================
# TARGET
# ===========================================

multiclass_target = "match_result_3class"

# ===========================================
# CLEAN DATA
# ===========================================

numeric_columns = [

    "home_rest_days",
    "away_rest_days",
    "rest_days_diff",

    "home_coach_tenure_days",
    "away_coach_tenure_days",
    "coach_tenure_diff",

    "home_points",
    "away_points",

    "home_position",
    "away_position",

    "points_diff",
    "position_diff",

    "points_diff_abs",
    "position_diff_abs",

    "home_elo",
    "away_elo",

    "elo_diff",

    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "attack_diff",
    "defense_diff",

    "combined_scoring",
    "combined_conceded",

    "total_balance",

    "home_draw_rate_last_10",
    "away_draw_rate_last_10",

    "draw_rate_diff",
    "draw_rate_sum",

    "home_advantage_strength",
    "away_advantage_strength",

    "strength_gap",

    "table_pressure_match",

    "round_number"
]

for col in numeric_columns:

    matches_features[col] = pd.to_numeric(
        matches_features[col],
        errors="coerce"
    )

multiclass_df = matches_features.dropna(
    subset=multiclass_features + ["match_result_3class"]
).copy()

# ===========================================
# TRAIN / TEST SPLIT
# ===========================================

train_df = multiclass_df[
    multiclass_df["season"] <= TRAIN_MAX_SEASON
].copy()

test_df = multiclass_df[
    multiclass_df["season"] >= TEST_MIN_SEASON
].copy()

# ===========================================
# X / Y
# ===========================================

X_train = train_df[multiclass_features]
y_train = train_df[multiclass_target]

X_test = test_df[multiclass_features]
y_test = test_df[multiclass_target]

# ===========================================
# MODEL
# ===========================================

xgb_model = XGBClassifier(

    objective="multi:softprob",

    num_class=3,

    n_estimators=400,

    max_depth=5,

    learning_rate=0.03,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="mlogloss"
)

# ===========================================
# TRAIN
# ===========================================

xgb_model.fit(
    X_train,
    y_train
)

# ===========================================
# PREDICTIONS
# ===========================================

predictions = xgb_model.predict(X_test)

probabilities = xgb_model.predict_proba(X_test)

# ===========================================
# REPORT
# ===========================================

print(
    classification_report(
        y_test,
        predictions
    )
)

# ===========================================
# RESULTS DF
# ===========================================

test_results = test_df.copy()

test_results["home_prob"] = probabilities[:, 0]

test_results["draw_prob"] = probabilities[:, 1]

test_results["away_prob"] = probabilities[:, 2]

test_results["prediction"] = predictions

# ===========================================
# OVERALL ACCURACY
# ===========================================

overall_accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Overall accuracy:",
    overall_accuracy
)

              precision    recall  f1-score   support

           0       0.53      0.78      0.63      1486
           1       0.25      0.02      0.04       873
           2       0.50      0.54      0.52      1102

    accuracy                           0.51      3461
   macro avg       0.43      0.45      0.40      3461
weighted avg       0.45      0.51      0.45      3461

Overall accuracy: 0.5131464894539151


In [29]:
future_matches = matches_features[
    (
        matches_features["status"] == "NS"
    ) &
    (
        matches_features["competition_id"] == 1
    ) &
    (
        matches_features["season"] == 2025
    )
].copy()

future_matches = build_future_matches(
    matches_df=matches_features,
    team_snapshot=team_snapshot,
    competition_name="Premier League",
    season=2025
)

future_matches["elo_diff"] = (
    future_matches["home_elo"] -
    future_matches["away_elo"]
).abs()

future_matches["attack_diff"] = (
    future_matches["home_rolling_scored"] -
    future_matches["away_rolling_scored"]
).abs()

future_matches["defense_diff"] = (
    future_matches["home_rolling_conceded"] -
    future_matches["away_rolling_conceded"]
).abs()

future_matches["combined_scoring"] = (
    future_matches["home_rolling_scored"] +
    future_matches["away_rolling_scored"]
)

future_matches["combined_conceded"] = (
    future_matches["home_rolling_conceded"] +
    future_matches["away_rolling_conceded"]
)

future_matches["total_balance"] = (
    future_matches["attack_diff"] +
    future_matches["defense_diff"]
)

future_matches["points_diff_abs"] = (
    future_matches["points_diff"]
).abs()

future_matches["position_diff_abs"] = (
    future_matches["position_diff"]
).abs()

future_matches["draw_rate_diff"] = (
    future_matches["home_draw_rate_last_10"] -
    future_matches["away_draw_rate_last_10"]
).abs()

future_matches["draw_rate_sum"] = (
    future_matches["home_draw_rate_last_10"] +
    future_matches["away_draw_rate_last_10"]
)

future_matches["home_advantage_strength"] = (
    future_matches["home_rolling_scored"] -
    future_matches["away_rolling_conceded"]
)

future_matches["away_advantage_strength"] = (
    future_matches["away_rolling_scored"] -
    future_matches["home_rolling_conceded"]
)

future_matches["strength_gap"] = (
    future_matches["home_advantage_strength"] -
    future_matches["away_advantage_strength"]
).abs()

future_matches["table_pressure_match"] = (
    future_matches["home_title_race"] +
    future_matches["away_title_race"] +
    future_matches["home_europe_race"] +
    future_matches["away_europe_race"] +
    future_matches["home_relegation_risk"] +
    future_matches["away_relegation_risk"]
)

future_matches["low_scoring_profile"] = (
    future_matches["combined_scoring"] <= 2.2
).astype(int)

future_matches["balanced_match_flag"] = (
    (
        future_matches["elo_diff"] <= 50
    ) &
    (
        future_matches["position_diff_abs"] <= 3
    ) &
    (
        future_matches["points_diff_abs"] <= 6
    )
).astype(int)
future_X = future_matches[
    multiclass_features
]

future_probabilities = (
    xgb_model.predict_proba(
        future_X
    )
)

future_matches["home_prob"] = (
    future_probabilities[:, 0]
)

future_matches["draw_prob"] = (
    future_probabilities[:, 1]
)

future_matches["away_prob"] = (
    future_probabilities[:, 2]
)

NameError: name 'build_future_matches' is not defined

In [ ]:
future_matches["top_diff"] = (
    future_matches[
        ["home_prob", "draw_prob", "away_prob"]
    ]
    .apply(
        lambda row:
        np.sort(row.values)[-1] -
        np.sort(row.values)[-2],
        axis=1
    )
)
future_matches[
    [
        "date",

        "home_team_name",
        "away_team_name",

        "home_prob",
        "draw_prob",
        "away_prob",

        "top_diff"
    ]
].sort_values(
    by="top_diff",
    ascending=True
)